# CelebA Simulation Results

The CelebA hair-colour design of Dance et al., run through the doubly robust MMD flow.
Three data seeds, both intervention arms per seed, one repeat each. Evaluation follows the
reference protocol: 2000 particles per arm, decoded at 256 px, scored against a fixed set of
2000 held-out reference images that is disjoint from the logged sample. Every number below
averages the two arms within a seed first, then reports **mean ± standard deviation across
the three data seeds**.

**The default configuration**, which is what the headline row reports:

| | |
|---|---|
| solver | projected Nystrom |
| nuisance fitting | no split |
| encoder | frozen SD-VAE (external and untrained, so there is no autoencoder split) |
| initialisation | `observed` — the logged rows of the arm being intervened on |
| particles | M = 2000 |
| observations | n_obs = 20000 |

Every later section changes **exactly one** of those and leaves the rest alone.

Two conventions worth stating once. FID is the legacy TorchVision Inception-v3 scorer, which
is the convention of the reference tables; clean-FID is a different scorer and never appears
in the same column. And CelebA has no confounding-strength parameter to sweep — the design
fixes the confounding through the conditional P(X = 1 | A) — so the w-sweep section of the
sibling notebooks does not exist here.

In [1]:
import sys
import pandas as pd

sys.path.append(".")
import analysis_tables as at

METRICS = at.METRICS["CelebA"]
d = at.load("CelebA")
default = at.default_rows(d)


def table(frame, by, order, labels, title, note="", metrics=METRICS,
          first_column="Setting", label_width=22):
    # One row per value of `by`, in the order given, in the house plain layout.
    rows = []
    for value in order:
        subset = frame[frame[by] == value]
        rows.append((labels[value],
                     [at.summarise(subset, column) for _, column in metrics]))
    return at.plain_table(rows, [label for label, _ in metrics], title=title, note=note,
                          first_column=first_column, label_width=label_width)


def cells_of(frame, by, order, labels, columns, headers, title, note="",
             first_column="Setting", label_width=22):
    # Same, but with an explicit column list; NaN means the column is unavailable.
    rows = []
    for value in order:
        subset = frame[frame[by] == value]
        values = []
        for column in columns:
            mean, sd = at.summarise(subset, column)
            values.append(None if pd.isna(mean) else (mean, sd))
        rows.append((labels[value], values))
    return at.plain_table(rows, headers, title=title, note=note, first_column=first_column,
                          label_width=label_width)


print("Coverage: every cell should hold 3 data seeds x 2 arms = 6 rows.")
print()
counts = d.groupby(["init", "dr_fit", "n_obs", "n_particles_requested"]).size()
for (init, fit, n_obs, n_particles), n_rows in counts.items():
    flag = "" if n_rows == 6 else "   <-- INCOMPLETE"
    print(f"  init={init:<11} {fit:<9} n_obs={n_obs:<6} M={n_particles:<5} "
          f"rows={n_rows:>2}{flag}")
print()
print(f"  total rows {len(d)},  solver(s) {sorted(d.solver.unique())},  "
      f"repeats {sorted(int(r) for r in d.repeat.unique())}")

Coverage: every cell should hold 3 data seeds x 2 arms = 6 rows.

  init=moment      nosplit   n_obs=20000  M=2000  rows= 6
  init=normal      nosplit   n_obs=20000  M=2000  rows= 6
  init=normal_raw  nosplit   n_obs=20000  M=2000  rows= 6
  init=observed    crossfit  n_obs=5000   M=2000  rows= 6
  init=observed    crossfit  n_obs=10000  M=2000  rows= 6
  init=observed    crossfit  n_obs=15000  M=2000  rows= 6
  init=observed    crossfit  n_obs=20000  M=2000  rows= 6
  init=observed    nosplit   n_obs=5000   M=2000  rows= 6
  init=observed    nosplit   n_obs=10000  M=2000  rows= 6
  init=observed    nosplit   n_obs=15000  M=2000  rows= 6
  init=observed    nosplit   n_obs=20000  M=250   rows= 6
  init=observed    nosplit   n_obs=20000  M=500   rows= 6
  init=observed    nosplit   n_obs=20000  M=1000  rows= 6
  init=observed    nosplit   n_obs=20000  M=2000  rows= 6

  total rows 84,  solver(s) ['projected_nystrom'],  repeats [0]


## 1. Headline

Our default run against the three DecFM rows, ported verbatim from the reference tables.
Lower is better in every column.

In [2]:
print(at.headline_table(d, "CelebA", at.DECFM_CELEBA))
print()
print()
print(at.plain_table(
    [("No flow (logged arm)", [at.summarise(default, c) for c in
                               ("sw2_observed", "fid_legacy_observed", "latent_sw2_initial")]),
     ("VAE round-trip",       [at.summarise(default, c) for c in
                               ("sw2_roundtrip", "fid_legacy_roundtrip")] + [None]),
     ("DR MMD Flow (ours)",   [at.summarise(default, c) for c in
                               ("sw2_flow", "fid_legacy_flow", "latent_sw2_flow")])],
    ["SW2", "FID", "Latent SW2"],
    title="CelebA, same runs: what the flow starts from   (n_obs = 20000, M = 2000)",
    note="No flow = the logged rows of the target arm, scored against the same references.\n"
         "VAE round-trip = those same images encoded and decoded once, with no flow at all;\n"
         "it has no latent row because it is not a set of particles."))

CelebA   (n_obs = 20000, M = 2000, seeds [0, 1, 2])

Method                SW2 ↓                FID ↓                Latent SW2 ↓
----------------------------------------------------------------------------
DecFM-Plugin          0.03272 ± 0.00183    11.497 ± 0.156       0.06220 ± 0.00071
DecFM-DR-I            0.03356 ± 0.00318    14.609 ± 0.525       0.06241 ± 0.00405
DecFM-DR-OT           0.03636 ± 0.00245    15.521 ± 1.307       0.06746 ± 0.00257
DR MMD Flow (ours)    0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136

Ours: projected Nystrom, no-split nuisance, no-split autoencoder, P_Y initialisation.
FID is the legacy TorchVision Inception-v3 scorer, matching the reference convention.


CelebA, same runs: what the flow starts from   (n_obs = 20000, M = 2000)

Method                SW2 ↓                FID ↓                Latent SW2 ↓
----------------------------------------------------------------------------
No flow (logged arm)  0.04960 ± 0.00130    11.082 ± 0.129     

The flow is best on both distributional metrics: pixel SW2 **0.02656** against the best
reference row's 0.03272 (19% lower), and latent SW2 **0.05129** against 0.06220 (18% lower).
FID **11.567** ties DecFM-Plugin's 11.497 and sits well below both DR variants (14.609 and
15.521).

The second table is the caveat that makes the FID column readable. The logged arm, before any
flow is run, already scores FID 11.08 — FID asks whether the decoded images are valid CelebA
faces, and the logged images are real photographs, so it barely registers the intervention at
all. SW2 is the column that measures whether the intervention was applied (0.0496 before the
flow, 0.0266 after); FID is the column that checks we did not wreck the images getting there.
The round-trip row shows the SD-VAE itself costs essentially nothing (SW2 0.04974 against the
logged 0.04960), so the reduction is transport, not decoder smoothing.

(The first table's footer is the shared helper's boilerplate. For CelebA, "no-split
autoencoder" means there is no autoencoder to split — the SD-VAE is external and frozen — and
the initialisation it calls P_Y is the `observed` arm, i.e. the logged rows of the arm being
intervened on.)

## 2. Ablation: nuisance fitting

No-split (the default) against cross-fitted nuisance estimation, everything else held at the
default configuration. Both arms of this comparison are complete: 4 sample sizes x 3 seeds x
2 arms, paired run for run.

In [3]:
subset = d[(d.init == "observed") & (d.n_particles_requested == 2000) & (d.n_obs == 20000)]
print(table(subset, "dr_fit", ["nosplit", "crossfit"],
            {"nosplit": "No split (default)", "crossfit": "Cross-fitted"},
            title="CelebA, nuisance fitting   (n_obs = 20000, M = 2000, 3 seeds)"))
print()
print()

# paired, run for run, on the shared (n_obs, data_seed, repeat, arm) key
key = ["n_obs", "data_seed", "repeat", "target_a"]
paired = d[(d.init == "observed") & (d.n_particles_requested == 2000)]
a = paired[paired.dr_fit == "nosplit"].set_index(key).sort_index()
b = paired[paired.dr_fit == "crossfit"].set_index(key).sort_index()
shared = a.index.intersection(b.index)
print(f"Paired differences (cross-fitted minus no-split), {len(shared)} matched runs:")
for label, column in METRICS:
    diff = b.loc[shared, column] - a.loc[shared, column]
    at_20k = diff[[i for i in shared if i[0] == 20000]]
    print(f"  {label:<12} max |diff| = {diff.abs().max():.2e} over all n_obs,"
          f"   {at_20k.abs().max():.2e} at n_obs = 20000")

CelebA, nuisance fitting   (n_obs = 20000, M = 2000, 3 seeds)

Setting               SW2 ↓                FID ↓                Latent SW2 ↓
----------------------------------------------------------------------------
No split (default)    0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136
Cross-fitted          0.02656 ± 0.00156    11.567 ± 0.086       0.05129 ± 0.00136


Paired differences (cross-fitted minus no-split), 24 matched runs:
  SW2          max |diff| = 6.64e-04 over all n_obs,   2.56e-05 at n_obs = 20000
  FID          max |diff| = 1.80e-02 over all n_obs,   3.84e-03 at n_obs = 20000
  Latent SW2   max |diff| = 2.83e-04 over all n_obs,   1.74e-05 at n_obs = 20000


The two fits are the same method here, to five decimal places. That is expected and not a bug:
CelebA's context-and-action kernel `k_S` is **exactly rank 4** — the cell is 2A + X with both A
and X binary — so the conditional mean embedding is essentially determined by the four cell
means and leaving a fold out barely changes it. Across the six paired runs at the default
n_obs = 20000 the largest difference on SW2 is **2.6e-05**; across all 24 paired runs it is
6.6e-04, at the smallest sample size where the cell counts are thinnest.

Cross-fitting is therefore free of cost on this design but also buys nothing, which is why the
default is the cheaper no-split fit. The weight diagnostics agree: effective sample size on the
arm is 7200 out of 10000 under both fits, the weights sum to 1 with no negative mass
(`sum_abs_v` = `sum_v` = 1.000), and the maximum weight is 6.25 (no split) against 6.37
(cross-fitted).

## 3. Ablation: number of observations

The logged sample size, at the default M = 2000 particles, no-split nuisance and `observed`
initialisation. The reference set is held fixed at 2000 images, so every row is scored against
the same target — only the amount of data feeding the DR target changes. All four sizes are
complete (3 seeds x 2 arms each).

In [4]:
subset = d[(d.init == "observed") & (d.dr_fit == "nosplit")
           & (d.n_particles_requested == 2000)]
sizes = [5000, 10000, 15000, 20000]
print(table(subset, "n_obs", sizes, {n: f"{n}" for n in sizes},
            title="CelebA, logged sample size   (M = 2000, no-split, 3 seeds)",
            first_column="n_obs"))
print()
print()
print(cells_of(subset, "n_obs", sizes, {n: f"{n}" for n in sizes},
               ["sw2_observed", "sw2_flow", "mean_transport"],
               ["SW2 (no flow)", "SW2 (flow)", "Transport"],
               title="CelebA, the same rows before and after the flow",
               note="Transport = mean L2 distance a particle travels in the standardised "
                    "latent space.\nIt is a diagnostic, not a metric to minimise, so ignore "
                    "the arrow on that column.",
               first_column="n_obs"))

CelebA, logged sample size   (M = 2000, no-split, 3 seeds)

n_obs                 SW2 ↓                FID ↓                Latent SW2 ↓
----------------------------------------------------------------------------
5000                  0.02996 ± 0.00245    11.589 ± 0.176       0.05674 ± 0.00145
10000                 0.02954 ± 0.00128    11.446 ± 0.129       0.05444 ± 0.00060
15000                 0.02705 ± 0.00138    11.593 ± 0.151       0.05160 ± 0.00113
20000                 0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136


CelebA, the same rows before and after the flow

n_obs                 SW2 (no flow) ↓      SW2 (flow) ↓         Transport ↓
---------------------------------------------------------------------------
5000                  0.05557 ± 0.00561    0.02996 ± 0.00245    5.325 ± 0.134
10000                 0.05576 ± 0.00501    0.02954 ± 0.00128    6.951 ± 0.147
15000                 0.05503 ± 0.00504    0.02705 ± 0.00138    8.288 ± 0.095
20000                

More data helps, steadily and modestly: pixel SW2 falls **0.02996 → 0.02656** and latent SW2
**0.05674 → 0.05129** over a fourfold increase in n_obs, an 11% and a 10% improvement. FID is
flat across the sweep (11.45 to 11.59, inside its own ±0.15 seed spread), consistent with the
reading above — more data sharpens the interventional target, it does not change whether the
decoded images are valid faces.

The second table shows why this is a real effect and not drift in the reference set: the
no-flow baseline is essentially constant (0.0556, 0.0558, 0.0550, 0.0496), so the improvement
comes from the flow, and mean transport grows 5.3 → 9.1 as the larger logged sample gives the
DR target a richer support to pull towards.

## 4. Ablation: number of particles

M = 250, 500, 1000, 2000 at the top sample size, everything else at the default.

**This section must be read against count-matched floors.** Sliced Wasserstein is strongly
sample-size biased: two independent draws of the *same* law score SW2 0.081 at n = 250 and
0.040 at n = 1000. The plain floor is a fixed 1000-vs-1000 number and does not move with M, so
comparing a 250-particle flow to it would credit the flow for a purely statistical effect. The
floor columns below are instead recomputed at each row's own scoring count. A ratio **below 1**
is not an improvement — it means the particle cloud is more regular than an honest independent
sample of the same size, i.e. over-regularisation.

In [5]:
subset = d[(d.init == "observed") & (d.dr_fit == "nosplit") & (d.n_obs == 20000)]
Ms = [250, 500, 1000, 2000]
labels = {M: f"M = {M}" for M in Ms}

print(cells_of(subset, "n_particles_requested", Ms, labels,
               ["sw2_flow", "sw2_floor_matched", "latent_sw2_flow",
                "latent_sw2_floor_matched"],
               ["SW2", "SW2 floor", "Latent SW2", "Latent floor"],
               title="CelebA, particle count vs the COUNT-MATCHED floor   (n_obs = 20000)",
               note="Floor = two independent halves of the reference set, both cut to the "
                    "count this row\nwas scored at (n_floor_matched below). The floor cannot "
                    "be enlarged past half the\nreference set, so it saturates at 1000. Floor "
                    "columns are references, not objectives.",
               first_column="Particles"))
print()
for M in Ms:
    row = subset[subset.n_particles_requested == M]
    n_matched = int(row.n_floor_matched.iloc[0])
    ratio = at.summarise(row, "sw2_flow")[0] / at.summarise(row, "sw2_floor_matched")[0]
    lat = (at.summarise(row, "latent_sw2_flow")[0]
           / at.summarise(row, "latent_sw2_floor_matched")[0])
    print(f"  M = {M:<5} scored at {int(row.n_flow_eval.iloc[0]):>4} images,"
          f" floor at {n_matched:>4}   SW2 / floor = {ratio:.2f},"
          f"   latent SW2 / floor = {lat:.2f}")
print()
print()
print(cells_of(subset, "n_particles_requested", Ms, labels,
               ["fid_legacy_flow_half", "fid_legacy_floor", "fid_legacy_flow"],
               ["FID (matched)", "FID floor", "FID (raw)"],
               title="CelebA, particle count in FID   (n_obs = 20000)",
               note="FID (matched) scores the row's first 1000 decoded images against the "
                    "second reference half,\nwhich is exactly the floor's count; it is '--' "
                    "where the row holds fewer than 1000 images.\nFID (raw) scores all M "
                    "images and is NOT comparable across M: FID is biased upward at\nsmall "
                    "sample size, which is most of the 75.7 -> 11.6 trend in that column.",
               first_column="Particles"))

CelebA, particle count vs the COUNT-MATCHED floor   (n_obs = 20000)

Particles             SW2 ↓                SW2 floor ↓          Latent SW2 ↓         Latent floor ↓
---------------------------------------------------------------------------------------------------
M = 250               0.07024 ± 0.00169    0.08079 ± 0.00708    0.13115 ± 0.00103    0.15177 ± 0.00247
M = 500               0.04838 ± 0.00138    0.05685 ± 0.00188    0.09404 ± 0.00087    0.10897 ± 0.00167
M = 1000              0.03555 ± 0.00222    0.04001 ± 0.00240    0.06950 ± 0.00181    0.07744 ± 0.00164
M = 2000              0.02656 ± 0.00155    0.04001 ± 0.00240    0.05129 ± 0.00136    0.07744 ± 0.00164

Floor = two independent halves of the reference set, both cut to the count this row
was scored at (n_floor_matched below). The floor cannot be enlarged past half the
reference set, so it saturates at 1000. Floor columns are references, not objectives.

  M = 250   scored at  250 images, floor at  250   SW2 / floor = 

The raw columns look like a clean monotone gain — SW2 0.0702 → 0.0266, FID 75.7 → 11.6 — and
almost all of that is sample-size bias on both metrics. Against the count-matched floor the
picture is the opposite: at M = 250, 500 and 1000, where the counts match exactly, the flow
sits **below** its floor (SW2 ratios 0.87, 0.85, 0.89; latent 0.86, 0.86, 0.90). Small-M clouds
are over-regularised, and SW2 rewards that.

FID is the metric that penalises it, and it is the one to read here. Count-matched at 1000
images, FID goes **22.54 at M = 1000 → 19.96 at M = 2000**, against a split-half floor of
19.09 and a logged-arm value of 19.43. Only at M = 2000 is the flow essentially at the validity
floor. Mean transport tells the same story from the other side — 34.5, 23.4, 15.0, 9.1 as M
grows: with M much smaller than the support, the energy flow behaves as a quantiser and drags
particles far off the face manifold to sit on local centroids.

One honest limit: the matched floor cannot be enlarged past half the reference set, so at
M = 2000 the flow row is scored at 2000 images against a 1000-image floor. Its SW2 ratio of
0.66 therefore mixes over-regularisation with a residual count gap and should not be quoted as
a number. The count-matched FID row is the reliable reading at M = 2000.

## 5. Ablation: initialisation

Four starting clouds, everything else at the default. All four are complete (3 seeds x 2 arms).

A word on what the arms mean, because CelebA's are not the same as the other datasets'. The
flow runs on **standardised** latents. A `torch.randn` drawn in those coordinates — the arm
called `normal` — therefore reproduces the pooled latents' mean and unit per-coordinate
variance *by construction*: it is a diagonal moment match, not an uninformative start.
`normal_raw` draws N(0, I) in the **raw** encoder space and pushes it through the same
standardising map, which is what `normal` already means on the datasets that do not standardise
— so `normal_raw` is the genuinely uninformative arm, and the one comparable with the other
notebooks' `normal`. `moment` is the empirical N(mu, Sigma) of the flow's own target support.

In [6]:
subset = d[(d.dr_fit == "nosplit") & (d.n_obs == 20000)
           & (d.n_particles_requested == 2000)]
arms = ["observed", "moment", "normal", "normal_raw"]
labels = {"observed": "observed (default)", "moment": "moment N(mu, Sigma)",
          "normal": "normal (standardised)", "normal_raw": "normal_raw (raw enc.)"}
print(table(subset, "init", arms, labels,
            title="CelebA, initialisation   (n_obs = 20000, M = 2000, 3 seeds)",
            note="normal is a diagonal moment match in the flow's own coordinates;\n"
                 "normal_raw is the genuinely uninformative start.", label_width=24))
print()
print()
print(cells_of(subset, "init", arms, labels,
               ["latent_sw2_initial", "latent_sw2_flow", "mean_transport"],
               ["Latent SW2 start", "Latent SW2 end", "Transport"],
               title="CelebA, where each start begins and where it ends IN LATENT SPACE",
               note="The floor for the latent columns is 0.0774 (two independent halves of "
                    "the reference set).\nStart and Transport are diagnostics; the arrow "
                    "applies to the end column only.", label_width=24))

CelebA, initialisation   (n_obs = 20000, M = 2000, 3 seeds)

Setting                 SW2 ↓                FID ↓                Latent SW2 ↓
------------------------------------------------------------------------------
observed (default)      0.02656 ± 0.00155    11.567 ± 0.085       0.05129 ± 0.00136
moment N(mu, Sigma)     0.03780 ± 0.00040    211.398 ± 1.631      0.05131 ± 0.00153
normal (standardised)   0.08618 ± 0.00134    292.896 ± 1.772      0.05417 ± 0.00158
normal_raw (raw enc.)   0.12753 ± 0.00144    314.482 ± 1.147      0.07028 ± 0.00112

normal is a diagonal moment match in the flow's own coordinates;
normal_raw is the genuinely uninformative start.


CelebA, where each start begins and where it ends IN LATENT SPACE

Setting                 Latent SW2 start ↓   Latent SW2 end ↓     Transport ↓
-----------------------------------------------------------------------------
observed (default)      0.07703 ± 0.00322    0.05129 ± 0.00136    9.076 ± 0.180
moment N(mu, Sigma)     0

This is the sharpest case in the project of a metric that cannot certify sample quality. All
four starts reach essentially the **same latent distribution** — latent SW2 0.0513, 0.0513,
0.0542, 0.0703 against a 0.0774 floor, so **all four are at or below it**, the uninformative
start included — while their **decoded images differ by more than an order of magnitude** in
FID: 11.6, 211.4, 292.9, 314.5. The latent metric clears every arm, including the three that
decode above FID 200. The space the flow optimises in is blind to the difference; only decoded
outcome space sees it. (As in section 4, that floor is a 1000-image number while these rows are
scored at 2000, so the margin below it mixes over-regularisation with a residual count gap and
is not a quality margin; the point here is only that no arm is flagged.)

Pixel SW2 ranks them the same way the eye would (0.0266, 0.0378, 0.0862, 0.1275) and the
uninformative arm is far worse than running no flow at all (0.0496). Transport separates them
cleanly too — 9.1, 16.9, 40.9, 65.4 — so the Gaussian starts travel a long way to arrive
somewhere latent-equivalent and pixel-catastrophic. Matching a mean embedding does not put
particles on the decoder's manifold, and matching the first two moments of the target support
does not either.

The practical consequence for this pipeline: **initialise at the logged arm**, and never
validate an image flow on the metric it was optimised against.

## Summary

- **We beat the reference on the metrics that measure the intervention.** Pixel SW2
  0.02656 ± 0.00155 against the best DecFM row's 0.03272 ± 0.00183, latent SW2
  0.05129 ± 0.00136 against 0.06220 ± 0.00071. FID 11.567 ± 0.085 ties DecFM-Plugin (11.497)
  and clears both DR variants (14.609, 15.521).
- **FID is a validity check, not an intervention metric, on this design.** The logged arm
  already scores 11.08 before any flow runs, and the VAE round-trip costs nothing
  (SW2 0.04974 against the logged 0.04960). Report SW2 for the intervention and FID alongside.
- **Cross-fitting changes nothing here.** Largest paired difference on SW2 is 2.6e-05 at the
  default n_obs = 20000 (6.6e-04 across all 24 paired runs). `k_S` is exactly rank 4, so the
  CME is essentially determined and leaving a fold out cannot move it. Keep the cheaper
  no-split fit.
- **More data helps steadily and modestly.** SW2 0.02996 → 0.02656 and latent SW2
  0.05674 → 0.05129 from n_obs = 5000 to 20000, while the no-flow baseline stays flat
  (0.0556 → 0.0496) and FID does not move (11.45 to 11.59).
- **Particle count is the real lever, and only the count-matched floor shows it.** At
  M = 250/500/1000 the flow sits below its own matched floor (SW2 ratios 0.87/0.85/0.89) —
  over-regularisation, not accuracy. Count-matched FID falls 22.54 (M = 1000) → 19.96
  (M = 2000) against a 19.09 floor, and transport drops 34.5 → 9.1. M = 2000 is the first
  setting at the validity floor.
- **Initialisation is where the evaluation space lies to you.** All four starts land within
  0.0513–0.0703 latent SW2 (floor 0.0774) while decoded FID spans 11.6 to 314.5. Initialise at
  the logged arm.
- **No confounding-strength section:** the CelebA design has no swept w parameter.